# Prepare the data for the differential accessibility notebook

This notebook runs once, on Engaging, and writes the small files in `03_differential/data/` that the
Colab notebook (`differential_atac.ipynb`) downloads. Students don't need to run it.

**Inputs**

- the nf-core/atacseq results (Step 2): peak counts, peak annotation, filtered BAMs, QC report
- whole-genome sequencing (WGS) of both lines from the same paper, aligned in `prep/wgs/`
- AmpliconArchitect results from that WGS, in `prep/aa/`

**Outputs** (in `data/`)

| File | What it is |
|---|---|
| `peaks.tsv.gz` | one row per consensus peak: position, GC, genomic feature, nearest gene, read counts for the 4 samples, copy number of each line, region class |
| `library_qc.tsv` | per-sample QC from the pipeline |
| `copy_number_1mb.tsv.gz` | copy number in 1 Mb bins, for the genome-wide plots: from WGS (each line) and from ATAC reads between peaks (each sample) |
| `bins_10kb.tsv.gz` | ATAC reads (in peaks / between peaks) and WGS depth in 10 kb bins: the MYC amplicon plus a sample of the rest of the genome |
| `myc_region_genes.tsv` | gene positions around MYC, for the amplicon plot |
| `amplicon_segments.tsv` | AmpliconArchitect copy-number segments of the MYC amplicon in each line |
| `peaks_200bp.fa.gz` | 200 bp of DNA sequence around each peak center, for HOMER |
| `fragment_gc.tsv.gz` | fragments per sample by location (peak, near a peak, background), length and GC, in regions with equal copy number, for the GC check |

Run it with the `R (fastq2pheno)` kernel, or as a job: `sbatch run_prepare.sbatch`.

## Setup

In [1]:
suppressPackageStartupMessages(library(dplyr))

pipeline <- "../../02_pipeline/results"
consensus <- file.path(pipeline, "bwa/merged_library/macs2/narrow_peak/consensus")
genome_fa <- file.path(pipeline, "genome/genome.fa")
# the gene annotation the pipeline used, from 02_pipeline/local.config (not in git; see local.config.example)
gtf <- sub(".*['\"](.+)['\"].*", "\\1", grep("^\\s*gtf\\s*=", readLines("../../02_pipeline/local.config"), value = TRUE))

samples <- c("COLO320DM_REP1", "COLO320DM_REP2", "COLO320HSR_REP1", "COLO320HSR_REP2")
atac_bams <- file.path(pipeline, "bwa/merged_library", paste0(samples, ".mLb.clN.sorted.bam"))
wgs_bams <- c(DM = "wgs/COLO320DM.bam", HSR = "wgs/COLO320HSR.bam")

out <- "../data"
work <- "work"              # intermediate files (bedtools output), kept so reruns are fast
dir.create(out, showWarnings = FALSE); dir.create(work, showWarnings = FALSE)
main_chroms <- paste0("chr", c(1:22, "X"))

# run a shell command unless its output file already exists
run_once <- function(cmd, output) {
    if (!file.exists(output) || file.size(output) == 0) stopifnot(system(cmd) == 0)
}

## 1. Peaks: counts, GC content, annotation

In [2]:
fc <- read.delim(file.path(consensus, "consensus_peaks.mLb.clN.featureCounts.txt"), comment.char = "#", check.names = FALSE)
counts <- fc[, 7:10]
names(counts) <- sub(".mLb.clN.sorted.bam", "", names(counts))

peaks <- data.frame(peak = fc$Geneid, chr = fc$Chr, start = fc$Start, end = fc$End, width = fc$Length) |>
    bind_cols(counts[, samples]) |>
    filter(chr %in% main_chroms)
nrow(peaks)

[1] 107271

In [3]:
# GC content of each peak
bed <- file.path(work, "peaks.bed")
write.table(peaks[, c("chr", "start", "end", "peak")], bed, sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
run_once(paste("bedtools nuc -fi", genome_fa, "-bed", bed, "| cut -f4,8,9,11,13 >", file.path(work, "peaks_nuc.txt")),
         file.path(work, "peaks_nuc.txt"))
nuc <- read.delim(file.path(work, "peaks_nuc.txt"))
names(nuc) <- c("peak", "C", "G", "N", "length")
peaks$gc <- with(nuc, (C + G) / (length - N))[match(peaks$peak, nuc$peak)]

# genomic feature and nearest gene, from the pipeline's HOMER annotation
ann <- read.delim(file.path(consensus, "consensus_peaks.mLb.clN.annotatePeaks.txt"), check.names = FALSE)
ann <- ann[match(peaks$peak, ann[[1]]), ]
peaks$feature <- sub(" \\(.*", "", ann$Annotation)
peaks$tss_distance <- ann$`Distance to TSS`
gene_id <- sub("\\..*", "", ann$`Nearest PromoterID`)

# gene names and types from the GTF
gene_lines <- system(paste("zcat -f", gtf, "| awk -F'\\t' '$3 == \"gene\"' | cut -f1,4,5,9"), intern = TRUE)
genes <- data.frame(
    chr   = sub("\t.*", "", gene_lines),
    start = as.numeric(sapply(strsplit(gene_lines, "\t"), `[`, 2)),
    end   = as.numeric(sapply(strsplit(gene_lines, "\t"), `[`, 3)),
    id    = sub('.*gene_id "([^".]+).*', "\\1", gene_lines),
    name  = sub('.*gene_name "([^"]+)".*', "\\1", gene_lines),
    type  = sub('.*gene_type "([^"]+)".*', "\\1", gene_lines))
peaks$gene <- genes$name[match(gene_id, genes$id)]
peaks$gene_type <- genes$type[match(gene_id, genes$id)]
table(peaks$feature)


        exon   Intergenic       intron promoter-TSS          TTS 
        3909        31826        52391        14992         4153 

## 2. Copy number from WGS

Reads per 10 kb bin in each line's WGS, relative to that line's genome-wide median, corrected for GC
content (a loess fit of depth against bin GC). Bins with almost no reads in either line (satellite repeats
around centromeres, which can't be mapped uniquely) are left out.

In [4]:
chrom_sizes <- file.path(work, "chrom.sizes")
run_once(paste0("grep -P '^chr([0-9]+|X)\\t' ", genome_fa, ".fai | cut -f1,2 > ", chrom_sizes), chrom_sizes)
bins_bed <- file.path(work, "bins_10kb.bed")
run_once(paste("bedtools makewindows -g", chrom_sizes, "-w 10000 -i srcwinnum | sort -k1,1 -k2,2n >", bins_bed), bins_bed)

# WGS reads per bin (full-depth BAMs: this takes about an hour)
wgs_counts_file <- file.path(work, "wgs_10kb_counts.txt")
run_once(paste("bedtools multicov -p -q 30 -bams", paste(wgs_bams, collapse = " "), "-bed", bins_bed, ">", wgs_counts_file), wgs_counts_file)
bins_nuc <- file.path(work, "bins_10kb_nuc.txt")
run_once(paste("bedtools nuc -fi", genome_fa, "-bed", bins_bed, "| cut -f4,8,9,11,13 >", bins_nuc), bins_nuc)

wgs <- read.delim(wgs_counts_file, header = FALSE, col.names = c("chr", "start", "end", "bin", "DM", "HSR"))
bn <- read.delim(bins_nuc); names(bn) <- c("bin", "C", "G", "N", "length")
stopifnot(all(wgs$bin == bn$bin))
wgs$gc <- with(bn, (C + G) / pmax(length - N, 1))
wgs$n_fraction <- bn$N / bn$length

In [5]:
# depth relative to the genome median, GC-corrected, per line
gc_corrected_log2 <- function(depth, gc, use) {
    log_depth <- log2(depth / median(depth[use]) + 1e-3)
    fit <- loess(y ~ g, data = data.frame(y = log_depth[use], g = gc[use])[sample(sum(use), 50000), ], span = 0.5, family = "symmetric")
    corrected <- log_depth - predict(fit, newdata = data.frame(g = pmin(pmax(gc, 0.3), 0.6)))
    corrected - median(corrected[use])
}
set.seed(1)
mappable <- wgs$n_fraction < 0.01 & wgs$DM > 0.2 * median(wgs$DM) & wgs$HSR > 0.2 * median(wgs$HSR)
not_amplified <- mappable & wgs$DM < 2 * median(wgs$DM) & wgs$HSR < 2 * median(wgs$HSR)
wgs$log2_DM  <- gc_corrected_log2(wgs$DM,  wgs$gc, not_amplified)
wgs$log2_HSR <- gc_corrected_log2(wgs$HSR, wgs$gc, not_amplified)
wgs$log2_DM[!mappable] <- NA; wgs$log2_HSR[!mappable] <- NA

# copy number of each peak = its 10 kb bin, as a ratio to the genome average (1 = typical, 100 = 100x)
peak_bin <- paste0(peaks$chr, "_", floor((peaks$start + peaks$end) / 2 / 1e4) + 1)
peaks$copy_ratio_DM  <- round(2^wgs$log2_DM[match(peak_bin, wgs$bin)], 3)
peaks$copy_ratio_HSR <- round(2^wgs$log2_HSR[match(peak_bin, wgs$bin)], 3)
summary(peaks$copy_ratio_DM)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
  0.194   0.953   1.021   1.633   1.412 192.010     186 

In [6]:
# 1 Mb bins for the genome-wide plot (median of the 10 kb bins); written in section 5, with the ATAC estimates added
wgs$bin_1mb <- floor(wgs$start / 1e6)
cn_1mb <- wgs |>
    filter(!is.na(log2_DM)) |>
    group_by(chr, bin_1mb) |>
    summarize(start = first(bin_1mb) * 1e6, log2_DM = median(log2_DM), log2_HSR = median(log2_HSR), .groups = "drop") |>
    select(chr, start, log2_DM, log2_HSR)
nrow(cn_1mb)

[1] 2917

## 3. Amplicons (AmpliconArchitect) and region classes

AmpliconArchitect reports copy-number segments for every amplicon it finds. The MYC amplicon is the one
containing MYC in each line; it is made of pieces of chr8, chr3, chr13 and chr16 joined together. Each
peak gets one region class:

- **MYC amplicon, shared**: in the MYC amplicon, DM and HSR copy numbers within 2-fold;
- **MYC amplicon, differential**: in the MYC amplicon, more than 2-fold apart;
- **other amplicon**: in another amplicon with copy number of at least 8;
- **off-amplicon, CN different**: elsewhere, but the lines differ in copy number (1 Mb, log2 difference > 0.3);
- **off-amplicon, CN equal**: everything else.

In [7]:
read_segments <- function(line) {
    files <- Sys.glob(sprintf("aa/%s/%s_AA_results/%s_amplicon*_graph.txt", line, line, line))
    do.call(rbind, lapply(files, function(f) {
        seqs <- strsplit(grep("^sequence", readLines(f), value = TRUE), "\t")
        data.frame(line = sub("COLO320", "", line),
                   amplicon = sub(".*_(amplicon[0-9]+)_graph.txt", "\\1", f),
                   chr   = sub(":.*", "", sapply(seqs, `[`, 2)),
                   start = as.numeric(sub(".*:([0-9]+)[-+]$", "\\1", sapply(seqs, `[`, 2))),
                   end   = as.numeric(sub(".*:([0-9]+)[-+]$", "\\1", sapply(seqs, `[`, 3))),
                   copy_number = as.numeric(sapply(seqs, `[`, 4)))
    }))
}
segments <- rbind(read_segments("COLO320DM"), read_segments("COLO320HSR"))

# which amplicon contains MYC, in each line
myc_amplicon <- segments |> filter(chr == "chr8", start <= 127736000, end >= 127736000) |> distinct(line, amplicon)
segments$is_myc_amplicon <- paste(segments$line, segments$amplicon) %in% paste(myc_amplicon$line, myc_amplicon$amplicon)
myc_amplicon

line,amplicon
<chr>,<chr>
DM,amplicon2
HSR,amplicon4


In [8]:
# AmpliconArchitect copy number at each peak (NA = not in an amplicon in that line), and whether it's in the MYC amplicon
peak_mid <- (peaks$start + peaks$end) / 2
aa_copy_number <- function(which_line) {
    cn <- rep(NA_real_, nrow(peaks)); in_myc <- rep(FALSE, nrow(peaks))
    segs <- segments[segments$line == which_line, ]
    for (i in seq_len(nrow(segs))) {
        hit <- peaks$chr == segs$chr[i] & peak_mid >= segs$start[i] & peak_mid <= segs$end[i]
        cn[hit] <- pmax(cn[hit], segs$copy_number[i], na.rm = TRUE)
        in_myc[hit] <- in_myc[hit] | segs$is_myc_amplicon[i]
    }
    list(cn = cn, in_myc = in_myc)
}
aa_dm <- aa_copy_number("DM"); aa_hsr <- aa_copy_number("HSR")
peaks$aa_copy_number_DM <- round(aa_dm$cn, 1)
peaks$aa_copy_number_HSR <- round(aa_hsr$cn, 1)

in_myc <- aa_dm$in_myc | aa_hsr$in_myc
aa_max <- pmax(aa_dm$cn, aa_hsr$cn, na.rm = TRUE)
in_other_amplicon <- !in_myc & !is.na(aa_max) & aa_max >= 8
copy_log2_diff <- log2(peaks$copy_ratio_DM / peaks$copy_ratio_HSR)
fold_apart <- !is.na(copy_log2_diff) & abs(copy_log2_diff) > 1
cn_1mb_diff <- (cn_1mb$log2_DM - cn_1mb$log2_HSR)[match(paste(peaks$chr, floor(peaks$start / 1e6)), paste(cn_1mb$chr, cn_1mb$start / 1e6))]

peaks$region_class <- case_when(
    in_myc & !fold_apart ~ "MYC amplicon, shared",
    in_myc               ~ "MYC amplicon, differential",
    in_other_amplicon    ~ "other amplicon",
    !is.na(cn_1mb_diff) & abs(cn_1mb_diff) > 0.3 ~ "off-amplicon, CN different",
    TRUE                 ~ "off-amplicon, CN equal")
table(peaks$region_class, useNA = "ifany")


MYC amplicon, differential       MYC amplicon, shared 
                       183                        362 
off-amplicon, CN different     off-amplicon, CN equal 
                     20846                      85727 
            other amplicon 
                       153 

In [9]:
# MYC amplicon segments, for the amplicon plot (chr8 part)
amp_segments <- segments |> filter(is_myc_amplicon, chr == "chr8") |> select(line, chr, start, end, copy_number)
write.table(amp_segments, file.path(out, "amplicon_segments.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)

# genes around MYC
myc_genes <- genes |> filter(chr == "chr8", start < 128.3e6, end > 126.3e6, !grepl("^ENSG", name)) |> select(name, start, end, type)
write.table(myc_genes, file.path(out, "myc_region_genes.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
myc_genes

name,start,end,type
<chr>,<dbl>,<dbl>,<chr>
LRATD2,126552443,126558478,protein_coding
PCAT1,126556323,127419050,lncRNA
CASC19,127072694,127227541,lncRNA
PRNCR1,127079874,127092600,lncRNA
CASC8,127289808,127482139,lncRNA
POU5F1B,127322183,127420066,protein_coding
CCAT2,127400399,127402150,lncRNA
CASC11,127686343,127738987,lncRNA
MYC,127735434,127742951,protein_coding


## 4. Save the peak table

In [10]:
write.table(peaks, gzfile(file.path(out, "peaks.tsv.gz")), sep = "\t", quote = FALSE, row.names = FALSE)
str(peaks)

'data.frame':	107271 obs. of  19 variables:
 $ peak              : chr  "Interval_228" "Interval_229" "Interval_230" "Interval_231" ...
 $ chr               : chr  "chr1" "chr1" "chr1" "chr1" ...
 $ start             : int  10073 29000 180753 267740 733834 736537 778375 804507 826922 869686 ...
 $ end               : int  10659 29320 181950 268082 734063 736960 779231 804961 827987 870210 ...
 $ width             : int  587 321 1198 343 230 424 857 455 1066 525 ...
 $ COLO320DM_REP1    : int  37 5 66 25 1 16 349 26 168 64 ...
 $ COLO320DM_REP2    : int  44 3 101 27 4 16 371 25 201 87 ...
 $ COLO320HSR_REP1   : int  21 0 53 13 1 7 210 12 56 26 ...
 $ COLO320HSR_REP2   : int  38 1 76 16 8 19 289 17 126 58 ...
 $ gc                : num  0.577 0.769 0.678 0.503 0.41 ...
 $ feature           : chr  "Intergenic" "promoter-TSS" "Intergenic" "intron" ...
 $ tss_distance      : int  -1503 -394 -12142 1056 -26962 -24162 45 -12637 84 6955 ...
 $ gene              : chr  "DDX11L2" "MIR1302-2HG" "

## 5. ATAC and WGS in 10 kb bins

For reproducing the paper's measure (ATAC reads per copy of DNA). For every 10 kb bin: ATAC reads inside
consensus peaks, ATAC reads between peaks (peaks padded by 1 kb), WGS reads, GC and the number of peaks.
Only the MYC amplicon bins and a random 30,000 mappable bins elsewhere are kept, to keep the file small.

In [11]:
padded <- file.path(work, "peaks_padded.bed")
run_once(paste("bedtools slop -i", bed, "-g", chrom_sizes, "-b 1000 | sort -k1,1 -k2,2n | bedtools merge >", padded), padded)
in_peaks  <- file.path(work, "bins_10kb_in_peaks.bed");  run_once(paste("bedtools intersect -a", bins_bed, "-b", padded, ">", in_peaks), in_peaks)
off_peaks <- file.path(work, "bins_10kb_off_peaks.bed"); run_once(paste("bedtools subtract -a", bins_bed, "-b", padded, ">", off_peaks), off_peaks)
for (part in c("in_peaks", "off_peaks")) {
    f <- file.path(work, paste0("atac_10kb_", part, ".txt"))
    run_once(paste("bedtools multicov -p -q 30 -bams", paste(atac_bams, collapse = " "), "-bed", get(part), ">", f), f)
}
sum_by_bin <- function(f) {
    x <- read.delim(f, header = FALSE)
    s <- rowsum(as.matrix(x[, 5:8]), x$V4)
    m <- matrix(0, nrow(wgs), 4, dimnames = list(wgs$bin, samples)); m[rownames(s), ] <- s; m
}
atac_in <- sum_by_bin(file.path(work, "atac_10kb_in_peaks.txt"))
atac_off <- sum_by_bin(file.path(work, "atac_10kb_off_peaks.txt"))

**Copy number from ATAC.** Reads that fall between peaks are mostly background, and background scales with
copy number, so they give a rough copy-number estimate for each ATAC sample without WGS. Same recipe as the
WGS: reads per bp of between-peak sequence in each 10 kb bin (bins at least half between peaks), GC-corrected,
relative to the genome median, then the median per 1 Mb.

In [12]:
between_bp <- local({
    x <- read.delim(file.path(work, "atac_10kb_off_peaks.txt"), header = FALSE)
    bp <- rowsum(x$V3 - x$V2, x$V4)
    setNames(bp[match(wgs$bin, rownames(bp))], wgs$bin)
})
between_bp[is.na(between_bp)] <- 0
enough <- between_bp >= 5000
set.seed(1)
for (s in samples) {
    log2_cn <- gc_corrected_log2(atac_off[, s] / pmax(between_bp, 1), wgs$gc, not_amplified & enough)
    log2_cn[!(mappable & enough)] <- NA
    wgs[[paste0("log2_ATAC_", sub("COLO320", "", s))]] <- log2_cn
}
atac_1mb <- wgs |>
    filter(!is.na(log2_DM)) |>
    group_by(chr, bin_1mb) |>
    summarize(across(starts_with("log2_ATAC_"), ~ round(median(.x, na.rm = TRUE), 3)), .groups = "drop")
cn_1mb <- cn_1mb |> mutate(bin_1mb = start / 1e6) |> left_join(atac_1mb, by = c("chr", "bin_1mb")) |> select(-bin_1mb)
write.table(cn_1mb, gzfile(file.path(out, "copy_number_1mb.tsv.gz")), sep = "\t", quote = FALSE, row.names = FALSE)
round(cor(cn_1mb[, -(1:2)], use = "pairwise"), 2)

,log2_DM,log2_HSR,log2_ATAC_DM_REP1,log2_ATAC_DM_REP2,log2_ATAC_HSR_REP1,log2_ATAC_HSR_REP2
log2_DM,1.00,0.80,0.51,0.51,0.45,0.44
log2_HSR,0.80,1.00,0.44,0.44,0.52,0.52
log2_ATAC_DM_REP1,0.51,0.44,1.00,1.00,0.95,0.96
log2_ATAC_DM_REP2,0.51,0.44,1.00,1.00,0.95,0.96
log2_ATAC_HSR_REP1,0.45,0.52,0.95,0.95,1.00,0.99
log2_ATAC_HSR_REP2,0.44,0.52,0.96,0.96,0.99,1.00


In [13]:
bins <- data.frame(chr = wgs$chr, start = wgs$start, gc = round(wgs$gc, 3),
                   n_peaks = as.vector(table(factor(peak_bin, levels = wgs$bin))),
                   wgs_DM = wgs$DM, wgs_HSR = wgs$HSR)
colnames(atac_in) <- paste0("in_peaks_", sub("COLO320", "", samples))
colnames(atac_off) <- paste0("between_peaks_", sub("COLO320", "", samples))
bins <- cbind(bins, atac_in, atac_off)
bins$myc_amplicon <- bins$chr == "chr8" & bins$start >= 126.43e6 & bins$start + 1e4 <= 128.0e6

set.seed(1)
elsewhere <- which(!bins$myc_amplicon & mappable)
keep <- bins$myc_amplicon | seq_len(nrow(bins)) %in% sample(elsewhere, 30000)
write.table(bins[keep, ], gzfile(file.path(out, "bins_10kb.tsv.gz")), sep = "\t", quote = FALSE, row.names = FALSE)
table(myc_amplicon = bins$myc_amplicon[keep])

myc_amplicon
FALSE  TRUE 
30000   157 

## 6. Library QC and peak sequences

In [14]:
mqc <- file.path(pipeline, "multiqc/narrow_peak/multiqc_data")
first_value <- function(f) { x <- read.delim(file.path(mqc, f), check.names = FALSE); setNames(apply(x[, -1], 1, function(r) r[!is.na(r)][1]), x[[1]]) }
dedup <- read.delim(file.path(mqc, "mqc_picard_deduplication_1.txt"), check.names = FALSE, row.names = 1)
qc <- data.frame(
    sample = samples,
    unique_read_pairs_millions = round(dedup[samples, "Unique Pairs"] / 1e6, 1),   # before filtering
    duplicate_percent  = round(100 * dedup[samples, "Duplicate Pairs Nonoptical"] / (dedup[samples, "Unique Pairs"] + dedup[samples, "Duplicate Pairs Nonoptical"]), 1),
    peaks = first_value("multiqc_mlib_peak_count-plot.txt")[samples],
    FRiP  = round(first_value("multiqc_mlib_frip_score-plot.txt")[samples], 3))
write.table(qc, file.path(out, "library_qc.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)
qc

,sample,unique_read_pairs_millions,duplicate_percent,peaks,FRiP
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
COLO320DM_REP1,COLO320DM_REP1,41.7,66.1,72221,0.531
COLO320DM_REP2,COLO320DM_REP2,45.2,65.4,76409,0.570
COLO320HSR_REP1,COLO320HSR_REP1,27.7,62.8,66996,0.434
COLO320HSR_REP2,COLO320HSR_REP2,43.9,68.4,78182,0.493


In [15]:
# 200 bp of sequence around each peak center, for HOMER
centers <- data.frame(chr = peaks$chr, start = round(peak_mid) - 100, end = round(peak_mid) + 100, name = peaks$peak)
centers_bed <- file.path(work, "peak_centers_200bp.bed")
write.table(centers, centers_bed, sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
stopifnot(system(paste("bedtools getfasta -fi", genome_fa, "-bed", centers_bed, "-nameOnly | gzip >", file.path(out, "peaks_200bp.fa.gz"))) == 0)

## 7. GC check: fragment GC

For the "Is it PCR?" plot: the GC content of every fragment, from the filtered BAMs (properly paired,
MAPQ >= 30; duplicates are already removed), in 1 Mb windows where the WGS copy number of the two lines is equal
and neither is amplified (section 2). Each fragment is labeled *peak* (overlaps a consensus peak), *flank* (within
1 kb of one) or *background*, and counted by length (10 bp bins) and GC (0.02 bins). A few minutes per sample,
cached in `work/`.

In [16]:
# 1 Mb windows with equal copy number in the two lines, neither amplified
cn_equal <- cn_1mb[is.finite(cn_1mb$log2_DM) & is.finite(cn_1mb$log2_HSR) & abs(cn_1mb$log2_DM - cn_1mb$log2_HSR) < 0.15 &
                   cn_1mb$log2_DM < 1 & cn_1mb$log2_HSR < 1, ]
cn_equal_bed <- file.path(work, "cn_equal_1mb.bed")
write.table(data.frame(cn_equal$chr, as.integer(cn_equal$start), as.integer(cn_equal$start + 1e6)), cn_equal_bed,
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
flanks_bed <- file.path(work, "peaks_1kb.bed")
write.table(data.frame(peaks$chr, as.integer(pmax(0, peaks$start - 1000)), as.integer(peaks$end + 1000)), flanks_bed,
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
cat(nrow(cn_equal), "windows with equal copy number\n")

# per sample: fragments (read pairs, one record each) -> location, length and GC -> counts
fragment_gc <- function(s) {
    f <- file.path(work, paste0(s, ".fragment_gc.tsv"))
    run_once(paste0("bash -o pipefail -c ", shQuote(paste0(
        "samtools view -f 2 -F 0x904 -q 30 ", atac_bams[match(s, samples)], " ", paste(main_chroms, collapse = " "),
        " | awk -v OFS='\\t' '$9 > 0 && $9 <= 1000 {print $3, $4 - 1, $4 - 1 + $9}'",
        " | bedtools intersect -u -f 1.0 -a - -b ", cn_equal_bed,
        " | bedtools intersect -c -a - -b ", bed,
        " | bedtools intersect -c -a - -b ", flanks_bed,
        " | bedtools nuc -fi ", genome_fa, " -bed -",                    # adds pct_at, pct_gc, ... after the 5 columns
        " | awk -v OFS='\\t' 'NR > 1 {loc = $4 > 0 ? \"peak\" : ($5 > 0 ? \"flank\" : \"background\");",
        " n[loc OFS int(($3 - $2) / 10) * 10 OFS int($7 * 50) / 50]++}",
        " END {print \"location\", \"length\", \"gc\", \"n\"; for (k in n) print k, n[k]}' > ", f, ".tmp && mv ", f, ".tmp ", f))), f)
    cbind(sample = s, read.delim(f))
}
frag <- do.call(rbind, parallel::mclapply(samples, fragment_gc, mc.cores = 4))
frag <- frag[order(frag$sample, frag$location, frag$length, frag$gc), ]
write.table(frag, gzfile(file.path(out, "fragment_gc.tsv.gz")), sep = "\t", quote = FALSE, row.names = FALSE)
round(xtabs(n ~ sample + location, frag) / 1e6, 2)     # millions of fragments

2161 windows with equal copy number


                 location
sample            background flank peak
  COLO320DM_REP1        3.69  0.59 6.31
  COLO320DM_REP2        3.68  0.61 7.29
  COLO320HSR_REP1       3.04  0.49 3.79
  COLO320HSR_REP2       4.55  0.78 6.60

## Done

In [17]:
files <- list.files(out, full.names = TRUE)
print(data.frame(file = basename(files), MB = round(file.size(files) / 1e6, 1)))

# one archive for Google Drive: unpacks to data/ in the Colab working directory
old <- setwd(".."); tar("differential_atac_data.tar.gz", files = "data", compression = "gzip"); setwd(old)
cat("archive: ../differential_atac_data.tar.gz,", round(file.size("../differential_atac_data.tar.gz") / 1e6, 1), "MB\n")

                    file  MB
1  amplicon_segments.tsv 0.0
2       bins_10kb.tsv.gz 0.5
3 copy_number_1mb.tsv.gz 0.1
4     fragment_gc.tsv.gz 0.2
5         library_qc.tsv 0.0
6   myc_region_genes.tsv 0.0
7      peaks_200bp.fa.gz 6.8
8           peaks.tsv.gz 4.3


Warning message:
“invalid uid value replaced by that for user 'nobody'”


Warning message:
“invalid gid value replaced by that for user 'nobody'”


archive: ../differential_atac_data.tar.gz, 11.8 MB
